In [1]:
import os
import re
from kiwipiepy import Kiwi
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [2]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [3]:
db_fetched_data = """
신발은 무조건 자기가 신겠다고 뒤집어져서 등원길에 한바탕 전쟁을 치렀다.거꾸로 신겨놔도 절대 못 벗게 해서 결국 짝짝이로 어린이집 보냄.선생님께 죄송하다고 알림장 썼다. 슬슬 미운 두 살이 시작되려나 보다.
"""

In [4]:
kiwi = Kiwi()

tokens = kiwi.tokenize(db_fetched_data)
replacements = []

for i in range(len(tokens) - 1):
    t1 = tokens[i]    # 앞 단어
    t2 = tokens[i+1]  # 뒤에 오는 명사
    
    # 1. 뒤의 단어가 일반명사(NNG)인지 확인
    if t2.tag == "NNG":
        # 원문에서 해당 명사 바로 앞의 1글자짜리 단어 구역을 안전하게 추출
        # 형태소 분석기의 오류를 방지하기 위해 부사(MAG)와 동사/형용사(VV/VA) 활용형을 모두 포괄합니다.
        if t1.tag in ["VV", "VA", "MAG"]:
            original_word = db_fetched_data[t1.start:t1.end].strip()
            
            # 오해를 일으키는 핵심 원인인 '1글자 조사/수식어'인 경우만 저격
            if len(original_word) == 1:
                target_phrase = f"{original_word} {t2.form}"
                
                # 분석기가 찾아낸 원래 품사의 '원형(Lemma)'을 추출 (예: '잘' -> '자다', '갈' -> '가다')
                # 부사로 오진했더라도 '잘'은 기본 용언 원형을 '자다'로 매핑하여 힌트를 생성합니다.
                lemma = t1.form
                if original_word == "잘":
                    lemma = "자다"
                
                # 원문 글자는 100% 보존하면서, 괄호 안에 원형 힌트를 강제 주입 (범용성 핵심)
                fixed_phrase = f"{lemma} {t2.form}"
                replacements.append((target_phrase, fixed_phrase))

# 원본 문장의 손상 없이 안전하게 힌트 단어 주입 (중복 제거 후 치환)
for target, fixed in set(replacements):
    db_fetched_data = db_fetched_data.replace(target, fixed)

print(f"-> 변환된 최종 범용 원문 데이터:\n{db_fetched_data.strip()}")

-> 변환된 최종 범용 원문 데이터:
신발은 무조건 자기가 신겠다고 뒤집어져서 등원길에 한바탕 전쟁을 치렀다.거꾸로 신겨놔도 절대 못 벗게 해서 결국 짝짝이로 어린이집 보냄.선생님께 죄송하다고 알림장 썼다. 슬슬 미운 두 살이 시작되려나 보다.


In [5]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()

In [6]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 1000,
    GenParams.REPETITION_PENALTY: 1.2,
    GenParams.STOP_SEQUENCES: ["#", "Step", "주의사항"],  # 생성 흐름을 끊지 않도록 안전한 기호만 지정
}

extractor_model = ModelInference(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)

In [7]:
extract_prompt = f"""[Instruction]
당신은 육아 기록 전문가입니다. 주어진 [Data]의 각 문장을 순서대로 정밀 분석하여 아래 [Output Format] 양식에 맞춰 오직 핵심 라벨 결과만 깨끗하게 출력하세요. 원문의 글자 형태를 절대로 임의로 변형하거나 깨뜨리지 마십시오.

[Data]
{step1_insights}

[Output Format]
1. 문장원문: [문장 내용]
- 핵심어: 단어1, 단어2
- 감정: 슬픔, 기쁨
- 육아범주: 수면

[Output]
"""


In [8]:
try:
    extract_response = extractor_model.generate(prompt=extract_prompt)
    if 'results' in extract_response and len(extract_response['results']) > 0:
        step2_keywords = extract_response['results'][0].get('generated_text', '').strip()
    else:
        step2_keywords = str(extract_response).strip()
except Exception as e:
    print(f"1단계 실행 중 오류 발생: {e}")
    step2_keywords = ""

perfect_match_input = step2_keywords

In [9]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",  # 일기 생성 등 창의적 맥락에는 sample 방식이 자연스럽습니다.
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600,
    GenParams.REPETITION_PENALTY: 1.1,
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.8,
    GenParams.STOP_SEQUENCES: ["\n\n", "[END]"]
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)

In [10]:
diary_prompt = f"""너는 인스타그램에서 수만 명의 육아 맘들과 소통하며 오늘 하루 아이의 성장 기록을 다정하고 솔직하게 공유하는 대한민국 엄마이다.
제공된 [육아 데이터 블록]의 각 번호에 명시된 '핵심어', '감정', '육아범주' 라벨 정보만을 완벽하게 조합하여 아래 [출력 예시]의 형태처럼 부드러운 한국어 문장으로만 작성해라.

[출력 예시 - 이 톤앤매너와 정갈한 한국어 문법을 그대로 따르세요]
요즘 부쩍 잡고 서는 재미에 푹 빠졌는지 도통 누워서 잘 생각을 안 하네요.
침대 가드를 잡고 번쩍 서서 날 보며 배시시 웃는데 예쁘면서도 한숨이 절로 나오더라고요.
밤 11시가 다 되어서야 겨우 잠들었답니다.
낮에는 종일 졸졸 따라다니며 바짓가랑이를 붙잡고 울어서 정말 아무것도 할 수가 없었네요.
제대로 껌딱지 시기가 온 건지 정신적으로 참 지치고 피로감이 밀려오는 하루였답니다.

[작성 규칙 - 절대 준수]
1. 문장 개수 절대 일치: 제공된 데이터의 순서에 맞춰 정확히 5개의 한글 문장만 작성하고, 한 문장이 끝날 때마다 무조건 줄바꿈을 해라. 영어나 로마자 표기(jabando 등)는 절대로 쓰지 마라.
2. 오직 라벨 기반 매칭: '잡기', '서기', '밤 11시', '껌딱지 시기', '피로감' 등 라벨에 기재된 명사와 감정 단어들을 유기적으로 결합하여 문장을 완성해라. 
3. 주어 전면 생략: 문장 시작할 때 '우리 아기가~', '엄마는~' 같은 주어 단어는 절대 쓰지 마라.
4. 어미 통일: 모든 문장의 끝은 '~네요', '~했답니다', '~더라고요', '~나와요' 중 하나로만 끝맺어라. 국립국어원 맞춤법과 띄어쓰기를 완벽하게 준수해라.
5. 깨끗한 한글 출력: 문장 앞에 숫자 기호나 대괄호를 절대 붙이지 마라. 모든 종류의 이모지나 기호 이모티콘도 절대 포함하지 마라.
6. 마감 기호: 모든 문장 작성을 마친 바로 다음 줄에 무조건 [END] 라고만 출력해라.

[육아 데이터 블록]
{perfect_match_input}

[Diary]:"""


In [11]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) and writer_results else {}
raw_diary = first_writer_result.get('generated_text', '').strip() if isinstance(first_writer_result, dict) else str(first_writer_result).strip()

In [12]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

raw_lines = [line.strip() for line in raw_diary.split('\n') if line.strip()]

full_print_lines = []
for line in raw_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+|^\s*\[\d+[^\]]*\]', '', line).strip()
    
    line = re.sub(r'[\u4e00-\u9fff]', '', line)
    line = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?\'\"~%·]', '', line).strip()
    
    if line:
        full_print_lines.append(line)

def truncate_by_bytes(text, max_bytes=400):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    return text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore').strip() + "..."

final_lines = []
for line in full_print_lines:
    final_lines.append(truncate_by_bytes(line, 400))

final_diary = "\n".join(final_lines)


In [13]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 신발은 무조건 자기가 신겠다고 뒤집어져서 등원길에 한바탕 전쟁을 치렀다.거꾸로 신겨놔도 절대 못 벗게 해서 결국 짝짝이로 어린이집 보냄.선생님께 죄송하다고 알림장 썼다.
2. 슬슬 미운 두 살이 시작되려나 보다.


In [14]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 문장원문: 신발은 무조건 자기가 신겠다고 뒤집어져서 등원길에 한바탕 전쟁을 치렀다. 거꾸로 신겨놔도 절대 못 벗게 해서 결국 짝짝이로 어린이집 보냈음. 선생님께 죄송하다고 알림장 썼다.
- 핵심어: 신발, 전쟁, 짝짝이, 알림장
- 감정: 스트레스, 짜증
- 육아범주: 의류관리


In [15]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 신발은 꼭 제대로 신겨야 하는데 자꾸 뒤집어 신으려고 해서 등원길에 한바탕 전쟁을 치루고 말았네요.
[2번 일기]: 거꾸로 신겨놔도 절대 못 벗게 해서 결국 짝짝이 신겨서 어린이집에 보내야 했답니다.
[3번 일기]: 선생님이 보시면 너무 죄송할 것 같아서 알림장까지 써야 했더라고요.
[4번 일기]: 등원길에 전쟁을 치르고 나니까 너무 스트레스 받고 짜증이 나서 머리가 아파오네요.
[5번 일기]: 의류 관리는 쉽지 않은 것 같아요.

-> 최종 결과물 총 문장 수: 5줄
